# QuikDel Simulation


## Imports

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.colab import drive
import os
import time as t
import heapq
import json
import sys
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import ast
import requests
from geopy.distance import geodesic
import pickle
from shapely.geometry import shape
from collections import defaultdict

## MOST LIKELY NOT USE THESE
# !pip install memory-profiler
# !pip install psutil
# from memory_profiler import memory_usage
# import psutil

## Variables

In [ ]:
# City Data
city_name = "Columbus"
mini = True

In [ ]:
# Ratio data
superspot_hotspot_ratio = 10
min_children = superspot_hotspot_ratio - 3

## Directory Setup
In Colab, we connect to Google Drive.

In [ ]:
drive.mount('/content/gdrive/', force_remount=True)
personal_dir = "./gdrive/MyDrive/DeliverAI Data Folder/"
data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

# See directory content
data = os.listdir(data_dir)
print("Files in directory : ", data)

Mounted at /content/gdrive/
Files in directory :  ['Census Data', 'Original Location Data', 'Processed Location Data', 'Images', 'Hotspot Data', 'Q Tables', 'avg_hotspot_data.json', 'ExecutionTime_7_31.txt', 'UMST Graph']


## Load Data

#### Loading city data

In [ ]:
census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")

distance_matrix = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/distance_adjacency_matrix.npy")
time_matrix = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/time_adjacency_matrix.npy")

gh_df = pd.read_csv(data_dir + f'/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/graphhopper_dataframe.csv', dtype={'GEOID': 'string'}).drop(["Unnamed: 0"], axis=1)
num_hotspots = len(census_df.index)
print(num_hotspots)

FileNotFoundError: [Errno 2] No such file or directory: './gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Hotspot Data/Data-7-10/distance_adjacency_matrix.npy'

#### Loading Q-Table data

In [ ]:
def convert_str_to_int(d):
  return {int(k) if k.isdigit() else k: int(v) if isinstance(v, str) and v.isdigit() else v for k, v in d.items()}

with open(data_dir + "/Hotspot Data/map_geoid_index.json", 'r') as f:
    tract_to_index = json.load(f)
index_to_tract = {v: k for k, v in tract_to_index.items()}

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/index_to_Q.json", 'r') as f:
  index_to_Q = convert_str_to_int(json.load(f))

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_to_index.json", 'r') as f:
  Q_to_index = convert_str_to_int(json.load(f))

  for k, v in Q_to_index.items():
    Q_to_index[k] = convert_str_to_int(v)

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/index_to_Q_super.json", 'r') as f:
  index_to_Q_super = convert_str_to_int(json.load(f))

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_super_to_index.json", 'r') as f:
  Q_super_to_index = convert_str_to_int(json.load(f))

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/tract_group_lists.json", 'r') as f:
  tract_group_lists = convert_str_to_int(json.load(f))
  tract_group_lists[-1] = tract_group_lists['-1']
  del tract_group_lists['-1']

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/tract_super_dict.json", 'r') as f:
  tract_super_dict = convert_str_to_int(json.load(f))

Q_tables = np.load(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_boltzman.npz", allow_pickle=True)
Q_tables = {int(key): Q_tables[key] for key in Q_tables}

#### Creating Full/"Estimate" Distance and Time matrices
**Note:** The Estimate matrices will only be created if they do not already exist.

In [ ]:
if "estimate_distance_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data") or "estimate_time_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data"):
  # Import hotspots and order according to tract_to_index
  hotspot_df = gpd.read_file(data_dir + "/Hotspot Data/hotspot_data.geojson")
  hotspot_df['index_order'] = hotspot_df['GEOID'].map(tract_to_index)
  hotspot_df = hotspot_df.sort_values(by='index_order').reset_index(drop=True)
  hotspot_df = hotspot_df.drop(columns=['index_order'])


  # Turn hotspot dataframe into a matrix of coordinates getting the distance between them
  coords = hotspot_df['geometry'].apply(lambda geom: (geom.x, geom.y)).tolist()
  estimate_dist_matrix = np.zeros((len(coords), len(coords)))

  # Calculate pairwise geodesic distances
  for i, coord1 in enumerate(coords):
    for j, coord2 in enumerate(coords):
      estimate_dist_matrix[i, j] = geodesic(coord1, coord2).meters if i != j else 0.0

  # Add the average distance longer that actual roads take to the crow-flies distances
  dist_diff_matrix = distance_matrix - estimate_dist_matrix
  dist_diff_matrix = dist_diff_matrix[dist_diff_matrix < 100000000]
  avg_increase = dist_diff_matrix.mean()
  estimate_dist_matrix = estimate_dist_matrix + avg_increase

  # Get the speed of the distances from the distance/time matrices
  speed_matrix = distance_matrix / time_matrix
  np.nan_to_num(speed_matrix, 0)  # Set nan values to 0

  # Exclude values where speed is 1, as the vast majority (if not all) indicate edges that dont exist
  valid_speeds_mask = (speed_matrix != 1) & ~np.isnan(speed_matrix)

  # Mask speed matrix
  valid_speeds = speed_matrix[valid_speeds_mask]
  average_speed = np.mean(valid_speeds) # Get the mean of the matrix, this gives the average speed for each distance travelled

  estimate_time_matrix = estimate_dist_matrix / average_speed  # Use this and the full distance matrix to create the full time matrix

  np.save(data_dir + f"/Hotspot Data/estimate_distance_adjacency_matrix.npy", estimate_dist_matrix)
  np.save(data_dir + f"/Hotspot Data/estimate_time_adjacency_matrix.npy", estimate_time_matrix)

estimate_dist_matrix = np.load(data_dir + f"/Hotspot Data/estimate_distance_adjacency_matrix.npy")
estimate_time_matrix = np.load(data_dir + f"/Hotspot Data/estimate_time_adjacency_matrix.npy")

Below the full distance/time matrices are created and updated as necessary

In [ ]:
if "full_distance_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data") or "full_time_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data"):
  print("Creating new")
  full_dist_matrix = distance_matrix
  full_time_matrix = time_matrix

  np.save(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy", distance_matrix)
  np.save(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy", time_matrix)

else:
  print("Already exist")
  full_dist_matrix = np.load(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy")
  full_time_matrix = np.load(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy")

  # Copy distance/time_matrix data onto full, and save it again.
  for i in range(len(census_df.index)):
    for j in range(len(census_df.index)):
      if full_dist_matrix[i, j] == sys.maxsize and distance_matrix[i, j] != sys.maxsize:
        full_dist_matrix[i, j] = distance_matrix[i, j]
        full_time_matrix[i, j] = time_matrix[i, j]

  np.save(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy", full_dist_matrix)
  np.save(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy", full_time_matrix)

Already exist


NameError: name 'distance_matrix' is not defined


## Simulation Materials

#### Global Variables and Functions

In [ ]:
INTERVALS = [900, 1200, 1500, 1800, 2700, 3600, 5400, 7200, 10800, 14400]

In [ ]:
# Changed the directory
deliveries_dir = data_dir + f"/Deliveries"
try:
  os.mkdir(deliveries_dir)
except:
  print(deliveries_dir + " Already Exists")

In [ ]:
def write_to_full_matrix(start_node, end_node):
  if full_dist_matrix[start_node][end_node] == sys.maxsize or full_time_matrix[start_node][end_node] == sys.maxsize:
    from_row = gh_df[gh_df['GEOID'] == index_to_tract[start_node]]
    from_points = [[ from_row['x'].values[0], from_row['y'].values[0] ]]

    to_row = gh_df[gh_df['GEOID'] == index_to_tract[end_node]]
    to_points = [[ to_row['x'].values[0], to_row['y'].values[0] ]]

    gh_data = gh_query(from_points, to_points, keys)

    full_dist_matrix[start_node, end_node] = gh_data['distances'][0][0]
    full_time_matrix[start_node, end_node] = gh_data['times'][0][0]

    # print(f"Updated {start_node} to {end_node} with {gh_data['distances'][0][0]}")

    np.save(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy", full_time_matrix)
    np.save(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy", full_dist_matrix)



pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.colheader_justify', 'center')
pd.set_option('display.precision', 2)

def print_matrix(matrix):
    modified_matrix = np.where(matrix > 1000000, -1, matrix)

    index_labels = [index_to_tract[i][-4:] for i in range(len(matrix))]  # Setting row labels
    columns_labels = [index_to_tract[i][-4:] for i in range(len(matrix))]  # Setting col labels

    print(pd.DataFrame(modified_matrix, columns=columns_labels, index=index_labels))


##### Graphhopper Setup

In [ ]:
keys = [
  '4b0e0ccd-b94d-4a37-a8a6-d2fb8a4a66db',
]


key_ind = len(keys)-1  # Index in the list that determines which key is being used. Starts from last and works its way up because I feel cool when I do it that way

URL = "https://graphhopper.com/api/1/matrix"  # GraphHopper matrix API url
req_lim = 80  # Limit on number of requests in one go

# Generates GraphHopper query using provided params
def gh_query(from_point, to_point, keys: list):
  global key_ind

  # Do-while loop in python, runs query before checking
  while True:
    # Query to provide the key
    query = { "key": keys[key_ind] }

    # Payload to send rest of data
    payload = {
        "profile": "car",
        "from_points": from_point,
        "to_points": to_point,
        "out_arrays": ["distances","times"],
        "fail_fast": "true",
    }

    # Get response using POST
    response = requests.post(URL, json=payload, params=query)
    data = response.json()  # Store data as json

    # If there is a message, then something went wrong.
    if 'message' not in data.keys():
      return data  # No message so return data
    else:  # Assume message says key ran out
      if key_ind > 0:  # Make sure we don't exit bounds of list
        key_ind -= 1
      else:  # Otherwise we have run out of keys
        if 'Minutely API limit heavily violated' not in data['message']:
          print(data['message'])
        key_ind = len(keys)-1

  return None  # Function shouldn't reach this but it's here just in case since the only other return is in an if-statement

### Deliveries

#### Delivery Class

In [ ]:
class Delivery:
  id = 0
  def __init__(self, start_node, end_node, start_time, percent):
    # IMMUTABLE
    self.id = Delivery.id
    Delivery.id += 1
    self.start_node = start_node
    self.end_node = end_node
    self.start_time = start_time


    # MUTABLE
    self.current_node = self.start_node
    self.time_till_next_node = 0
    self.end_time = None
    self.in_transition = False
    self.is_primary = True
    self.next_node = None
    self.sharing_node = None
    self.completed = False
    self.successful = False
    self.num_car_changes = 2  # For initial change from PDC to CDV and for the final change to CDV to PDC
    self.sharing_with = None
    self.path = [self.start_node]
    self.distance_traveled = 0
    self.percent_max_dist = percent
    self.step = self.set_step()  # Make sure this is initialized before time_limit for dependency reasons
    self.time_limit = self.calculate_time_limit()

    # DEBUG VARIABLES (also mutable)
    self.did_share = False
    self.shared_with_final = None
    self.shared_with_when = None


  def calculate_time_limit(self):
    global INTERVALS

    path_time = full_time_matrix[self.start_node, self.end_node]

    for interval in INTERVALS:
      if path_time * 1.5 <= interval:
        return interval
    return float('inf')  # Realistically no delivery should reach this point.


  def check_direct_share(self):
    # Are census tracts in different groups?
    different_groups = tract_super_dict[self.start_node] != tract_super_dict[self.end_node]


    # Are the hotspots close enough for direct delivery?
    max_distance = max(
      np.max(estimate_dist_matrix),
      np.max(full_dist_matrix[full_dist_matrix != sys.maxsize]))

    close_proximity = full_dist_matrix[self.start_node, self.end_node] / max_distance <=  self.percent_max_dist / 100


    # Is the max sharing distance indended to be NEVER?
    never_share = self.percent_max_dist == 100


    # Must be ( in different groups AND close enough )  OR;
    #  maximum percentage of max distance travel == 100 (AKA do not share for all deliveries)
    return ( different_groups and close_proximity ) or never_share

  def set_step(self):
    ## GUIDE to STEPS:
    # - Step 1: Delivery is in sub-group, must travel to super-spot to reach destination.
    # - Step 2: Delivery is in super-spot level, must travel to next super-spot to reach destination.
    # - Step 3: Delivery is in sub-group, must travel to destination within the same group.
    # - Step 4: Delivery will travel directly from start to finish with no sharing potential.

    if self.check_direct_share():
      step_val = 4
    elif tract_super_dict[self.start_node] == tract_super_dict[self.end_node]:
      step_val = 3
    elif tract_super_dict[self.start_node] != tract_super_dict[self.end_node] and tract_super_dict[self.start_node] == self.start_node:
      step_val = 2
    else:
      step_val = 1

    return step_val

  def reset(self, p=-1):
    percent = self.percent_max_dist if p < 0 else p

    self.current_node = self.start_node
    self.time_till_next_node = 0
    self.end_time = None
    self.in_transition = False
    self.is_primary = True
    self.next_node = None
    self.sharing_node = None
    self.completed = False
    self.successful = False
    self.num_car_changes = 2
    self.sharing_with = None
    self.path = [self.start_node]
    self.distance_traveled = 0
    self.percent_max_dist = percent
    self.step = self.set_step()  # Make sure this is initialized before time_limit for dependency reasons
    self.time_limit = self.calculate_time_limit()

    # DEBUG VARIABLES
    self.did_share = False
    self.shared_with_final = None
    self.shared_with_when = None


  # How to determine delivery1 < delivery2
  def __lt__(self, other):
    # Compare start times
    if self.start_time != other.start_time:
      return self.start_time < other.start_time
    # Compare path lengths
    elif len(self.path) != len(other.path):
      return len(self.path) < len(other.path)
    # ID failsafe
    else:
      return self.id < other.id

  def __str__(self):
    return f"Delivery {self.id}\n" \
           f"Start Time: \t {self.start_time}\n" \
           f"Start Hotspot: \t {self.start_node}\n" \
           f"End Hotspot: \t {self.end_node}\n" \
           f"In Transition: \t {self.in_transition}\n" \
           f"Time limit: \t {self.time_limit}\n" \
           f"Time left: \t {self.time_till_next_node}\n" \
           f"Current Hotspot: {self.current_node}\n" \
           f"Next Hotspot: \t {self.next_node}\n" \
           f"Step: \t {self.step}\n" \
           f"Sharing With: \t {self.sharing_with.id if self.sharing_with is not None else None} \n" \
           f"Shared With Final: \t {self.shared_with_final} \n" \
           f"Shared With When: \t {self.shared_with_when} \n" \
           f"Path: \t \t {self.path}\n" \
           f"Completed: \t {self.completed}\n"

#### DeliveryList Class

In [ ]:
import copy  # Necessary to allow for deep copy creation

class DeliveryList:
  # distance_based = further, closer, <anything else for random>
  def __init__(self, distance_based:str, load:int, percent_max_dist:float, hours:int=1, peaks:list=[0.25, 0.75], sigma:float=10, gen_multi:bool=False, use_generated_paths:bool=False):
    self.distance_based = distance_based
    self.load = load
    self.hours = hours
    self.peaks = peaks
    self.sigma = sigma
    self.percent_max_dist = percent_max_dist
    self.gen_multi = gen_multi
    print("Generating ID...")
    self.id = DeliveryList.gen_deliveries_id(distance_based=self.distance_based, load=self.load, percent_max_dist=self.percent_max_dist,
                                             hours=self.hours, peaks=self.peaks, sigma=self.sigma, gen_multi=self.gen_multi)
    print("done.\n\nGenerating distribution...")
    self.distribution, self.loads = self.generate_distribution()
    print("done.\n\nGenerating deliveries...")
    self.deliveries = self.generate_deliveries(use_generated_paths)
    print("\ndone.\n\nSaving deliveries...")
    DeliveryList.save_deliveries(self)
    print("done.")


  # Function to generate delivery ID
  @staticmethod
  def gen_deliveries_id(distance_based:str, load:int, percent_max_dist:float, hours:int=1, peaks:list=[0.25, 0.75], sigma:float=10, gen_multi:bool=False, multi_id:int=0):
    global deliveries_dir

    if not gen_multi and multi_id > 0:
      raise ValueError("gen_multi must be True if multi_id is specified.")

    # Determine distance-based id
    match distance_based:
      case "further":
        base_code = "1"
      case "closer":
        base_code = "2"
      case "one_cluster":
        base_code = "3"
      case _:
        base_code = "0"

    # Construct initial id_str
    id_str = (
      f"{base_code}"
      f"-{load:04d}"
      f"-{hours:03d}"
      f"-{sigma:03d}"
      f"-{peaks}"
      f"-{percent_max_dist:03d}"
    )


    # List of already matching IDs
    list_matching_ids = [f for f in os.listdir(deliveries_dir) if f.startswith(id_str)] if gen_multi else []

    # The final id will be stored here
    final_id = id_str

    # If multi_id was specified
    if multi_id > 0:
      final_multi_id = multi_id-1
    # If the list is not empty:
    elif list_matching_ids:
      # multi_ids is a list of the terminating multi id number at the
      #   end of each of the ids
      multi_ids = (
        int(f.split('-')[-1])
        for f in list_matching_ids
        if '[' not in f.split('-')[-2] and f.split('-')[-1].isdigit()
      )

      # Determine the maximum multi-ID
      final_multi_id = max(multi_ids, default=0)
    else:
      return final_id

    # Update final_id based on max_multi_id
    final_id += f'-{final_multi_id + 1:02d}' if final_multi_id else '-01'

    return final_id


  # Function to generate distribution
  def generate_distribution(self):
    # mu for each peak
    mu_list = [peak * 60 * self.hours for peak in self.peaks]

    # x vals (one x value per minute)
    x = np.linspace(0, 60 * self.hours, 60 * self.hours)

    # Create individual normal distributions and sum them
    y_combined = np.zeros_like(x)  # Numpy array of 0's of len(x)
    for mu in mu_list:
      y_combined += (1 / (self.sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / self.sigma) ** 2)  # Performs distribution operation across array of x

    # Convert distribution into loads
    loads = [int((self.load * (self.sigma * np.sqrt(2 * np.pi))) * y) for y in y_combined]

    # IMPORTANT! What to return?
    return y_combined, loads


  def generate_deliveries(self, use_generated_paths:bool=False):
    with open(data_dir + "/avg_hotspot_data.json") as f:
      hotspot_data = json.load(f)

    valid_hotspots = [i for i in range(num_hotspots) if index_to_tract[i] in hotspot_data and hotspot_data[index_to_tract[i]]["distance"] != 0]

    deliveries = []
    with tqdm(total=sum(self.loads)) as pbar:
      for j in range(60*self.hours):  # For each minute
        for _ in range(self.loads[j]):
          while True:
            start = random.choice(valid_hotspots)
            end = random.choice(valid_hotspots)

            if use_generated_paths and full_time_matrix[start, end] == sys.maxsize:
              continue

            max_dist = max(
              np.max(estimate_dist_matrix),
              np.max(full_dist_matrix[full_dist_matrix != sys.maxsize]))
            tot = (estimate_dist_matrix[start][end] / max_dist)

            st_ed_eq = start != end

            if self.distance_based == "further":
              rand = random.uniform(0.10, 0.85)
              if tot > rand and st_ed_eq:
                break
            elif self.distance_based == "closer":
              rand = random.uniform(0.10, 1)
              if tot < rand and st_ed_eq:
                break
            else:
              if st_ed_eq:
                break

          # Generate time and delivery object
          time = random.randint(j * 60, (j + 1) * 60)  # Random second in the minute
          write_to_full_matrix(start, end)

          temp_delivery = Delivery(start, end, time, self.percent_max_dist)
          deliveries.append(temp_delivery)
          pbar.update(1)

    return deliveries


  # Function to save
  @staticmethod
  def save_deliveries(deliverylist, graph:bool=False):
    if not isinstance(deliverylist, DeliveryList):
      raise TypeError("Parameter deliverylist must be of type DeliveryList.")

    temp_dir = deliveries_dir

    if graph:
      temp_dir += "/Graphing"
      try:
        os.mkdir(temp_dir)
      except:
        pass

    deliveries_id = deliverylist.id
    if not graph:
      print("Saving to", temp_dir + f"/{deliveries_id}")
    with open(temp_dir + f"/{deliveries_id}", 'wb') as file:
      pickle.dump(deliverylist, file)


  # Function to load
  @staticmethod
  def load_deliveries(deliveries_id:str, graph:bool=False):
    temp_dir = deliveries_dir

    if graph:
      temp_dir += "/Graphing"
      try:
        os.mkdir(temp_dir)
      except:
        pass

    if not graph:
      print("Loading from", temp_dir + f"/{deliveries_id}")
    with open(temp_dir + f"/{deliveries_id}", 'rb') as file:
      deliveries = pickle.load(file)

    return deliveries


  # Function to reset all
  def reset_deliveries(self, percent_max_dist:float=-1):
    for d in self.deliveries:
      d.reset(percent_max_dist)

    self.percent_max_dist = percent_max_dist if percent_max_dist >= 0 else self.percent_max_dist
    self.id = DeliveryList.gen_deliveries_id(distance_based=self.distance_based, load=self.load, percent_max_dist=self.percent_max_dist,
      hours=self.hours, peaks=self.peaks, sigma=self.sigma, gen_multi=self.gen_multi)

  # Create and return a deep copy of self (completely independent of original)
  def copy(self):
    return copy.deepcopy(self)


  ## Functions to show various graphs below
  def plot_base_distribution(self):
    plt.scatter(y=self.distribution, x=list(range(len(self.distribution))), color="green")
    plt.title('Plot of Delivery Peaks Distribution')
    plt.xlabel('Time (minutes)')
    plt.ylabel('Weight')
    plt.show()

  def plot_load_distribution(self):
    plt.scatter(y=self.loads, x=list(range(len(self.loads))), color="green")
    plt.title('Plot of Delivery Load Distribution')
    plt.xlabel('Time (minutes)')
    plt.ylabel('Weight')
    plt.show()

  def plot_starttime_distribution(self):
    starttime_lst = [d.start_time for d in self.deliveries]
    plt.hist(starttime_lst, bins=10, color="lightgreen", edgecolor='grey', alpha=0.7)
    plt.title('Histogram of Delivery Starting Time')
    plt.xlabel('Distance (meters)')
    plt.ylabel('Frequency')
    plt.show()

  def plot_distance_distribution(self):
    distances = [full_dist_matrix[d.start_node][d.end_node] for d in self.deliveries]

    print("Average: ", np.mean(distances))
    plt.hist(distances, bins=10, color="lightgreen", edgecolor='grey', alpha=0.7)
    plt.title('Histogram of Delivery Distance')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.show()

  def plot_time_limits(self):
    d_limits = [d.time_limit for d in self.deliveries]
    d_base_times = [full_time_matrix[d.start_node, d.end_node] for d in self.deliveries]
    d_limits.sort()
    d_base_times.sort()
    plt.scatter(y=d_limits, x=list(range(len(self.deliveries))), label='Time Limit', color="darkgreen")
    plt.scatter(y=d_base_times, x=list(range(len(d_base_times))), label='Direct Travel Time', color="lightgreen")
    plt.title('Plot of Delivery Time Limits')
    plt.xlabel('Delivery (sorted by time limit)')
    plt.ylabel('Time (seconds)')
    plt.legend()
    plt.show()

    if len(d_limits) < len(d_base_times):
      print("Differing sizes!")

  def plot_distance_by_step(self):
    grouped_data = defaultdict(list)
    for d in self.deliveries:
      grouped_data[d.step].append(full_dist_matrix[d.start_node][d.end_node])

    labels = list(grouped_data.keys())
    boxplot_data = list(grouped_data.values())

    plot = plt.boxplot(boxplot_data, labels=labels)
    plt.title('Plot of Delivery Distance by Step')
    plt.xlabel('Step (1-4)')
    plt.ylabel('Distance (meters)')
    plt.show()

  # Function to check if two DeliveryLists are equal (have same parameters, not necessarily the same deliveries)
  def __eq__(self, other):
    if not isinstance(other, DeliveryList):
      return False

    return (self.distance_based == other.distance_based and
            self.load == other.load and
            self.hours == other.hours and
            self.peaks == other.peaks and
            self.sigma == other.sigma and
            self.percent_max_dist == other.percent_max_dist and
            np.array_equal(self.distribution, other.distribution))

  # Function to print overview of statistics
  def __str__(self):
    return f"DeliveryList {self.id}\n" \
           f"Delivery distance based on '{self.distance_based}'\n" \
           f"Load: {self.load}\n" \
           f"# Deliveries: {len(self.deliveries)}\n" \
           f"Hours: {self.hours}\n" \
           f"Peaks: {self.peaks}\n" \
           f"Sigma: {self.sigma}\n" \
           f"% of Maximum Distance to Share: {self.percent_max_dist}%\n"

###### COMPARE WITH OLD AS NECESSARY

In [ ]:
# mu1 = 15
# mu2 = 45
# sigma = 10
# load = 20

# x = np.linspace(0, 60, 60)
# y1 = (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x - mu1) / sigma)**2)
# y2 = (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x - mu2) / sigma)**2)

# y_combined = y1 + y2
# loads = [int((load*(sigma * np.sqrt(2*np.pi)))*x) for x in y_combined]

# print(sum(loads))

# # plt.scatter(range(len(y_combined)), y_combined)
# plt.scatter(range(len(loads)), loads)
# plt.title("Original")

### Simulation

In [ ]:
class Environment:

  def __init__(self, deliverylist:DeliveryList, num_hotspots, distance_matrix, time_matrix, Q_table):
    global INTERVALS
    global tract_super_dict

    self.clock = 0
    self.time_step = 10
    self.radius = 1000
    self.is_baseline = False
    self.num_hotspots = num_hotspots
    self.distance_matrix = distance_matrix
    self.time_matrix = time_matrix
    self.shares = 0
    self.Q = Q_table
    self.total_vehicles = 0
    ###self.vehicle_history = list() #Never Used???
    self.vehicles = [0 for i in range(self.num_hotspots)]
    self.edge_count = [[0 for i in range(self.num_hotspots)] for j in range(self.num_hotspots)]
    self.deliverylist = deliverylist
    self.deliveries = deliverylist.deliveries
    self.active_deliveries = list() # deliveries that have started their journey but not reached destination
    self.hotspot_deliveries = list() # deliveries that have reached a hotspot during their journey
    self.dormant_deliveries = [delivery for delivery in self.deliveries] # deliveries yet to start their journey
    self.num_car_changes = 0


    ##For metrics
    self.destination_success = np.zeros((num_hotspots,), dtype=int)
    self.source_success = np.zeros((num_hotspots,), dtype=int)
    self.order_rate = np.zeros((num_hotspots,), dtype=int)
    self.non_local_order_rate = np.zeros((num_hotspots,), dtype=int)



    self.scale_Q()
    #self.sort_actions()  # NOT USED RIGHT NOW
    #self.find_nearby_hotspots(self.radius)  # NOT USED RIGHT NOW

    #self.baseline_data = dict()
    #with open(data_dir + "/avg_intra_hotspot_location_data.json", 'r') as f:
      #self.baseline_data = json.load(f)

  # scale all Q values to the range 0-1
  def scale_Q(self):
    for _, local_Q in self.Q.items():
      for destination in range(len(local_Q)):
        for source in range(len(local_Q[destination])):
          MIN = min(local_Q[destination][source])
          MAX = max(local_Q[destination][source])
          local_Q[destination][source] = (local_Q[destination][source] - MIN) / (MAX - MIN)

  # NOT USED RIGHT NOW
  # def sort_actions(self):
  #   self.sorted_actions = [[list() for i in range(self.num_hotspots)] for j in range(self.num_hotspots)]

  #   for destination in range(self.num_hotspots):
  #     for source in range(self.num_hotspots):
  #       self.sorted_actions[destination][source] = np.argsort(self.Q[destination][source])


  # NOT USED RIGHT NOW
  # find all nearby hotspots for each hotspot
  # def find_nearby_hotspots(self, threshold):
  #   self.nearby_hotspots = [list() for i in range(self.num_hotspots)]

  #   for node1 in range(self.num_hotspots):
  #     for node2 in range(self.num_hotspots):
  #       if self.distance_matrix[node1][node2] <= threshold:
  #         self.nearby_hotspots[node1].append(node2)


  # assign deliveries to each hotspot
  def find_delivery_hotspot(self):
    self.delivery_hotspots = [list() for i in range(self.num_hotspots)]

    for delivery in self.active_deliveries:
      if delivery.in_transition and delivery.next_node != delivery.end_node:
        self.delivery_hotspots[delivery.next_node].append(delivery)
      else:
        self.delivery_hotspots[delivery.current_node].append(delivery)

  def run(self, sharing=True):
    while (self.active_deliveries or self.dormant_deliveries):
      self.step(sharing)

      # if (self.clock % 1000 == 0):                                                         ###DEBUG
      #   print("\n", "-"*40)                                                               ###DEBUG
      #   print("Clock: ", self.clock)                                                      ###DEBUG
      #   print(" || active_deliveries")                                                    ###DEBUG
      #   print(f"{len(self.active_deliveries)}\tactive_deliveries.", end="\t\t")
      #   print([{"id": d.id, "start": index_to_tract[d.end_node][-4:], "end": index_to_tract[d.start_node][-4:]} for d in self.active_deliveries])
      #   print(" || dormant_deliveries")
      #   print(f"{len(self.dormant_deliveries)}\tdormant_deliveries.", end="\t\t")
      #   print([{"id": d.id, "start": index_to_tract[d.end_node][-4:], "end": index_to_tract[d.start_node][-4:]} for d in self.dormant_deliveries])
      #   print("\n\n")
      # if(self.clock >= 50000):
      #   break



  def step(self,sharing=True):

    # add new deliveries
    self.add_deliveries()

    # increment global time
    self.clock += self.time_step

    # assign deliveries to hotspots
    self.find_delivery_hotspot()

    # set next nodes
    self.set_next_nodes(sharing)

    # empty the hotspot_deliveries list to prepare for next step
    self.hotspot_deliveries = list()

    to_remove = list()
    vehicle_debt = [0 for i in range(self.num_hotspots)]

    for delivery in self.active_deliveries:

      # move the delivery forward if its not waiting for any other delivery (4 cases where this happens)
      if delivery.in_transition or delivery.sharing_with is None or delivery.sharing_with.current_node==delivery.current_node or delivery.current_node != delivery.sharing_node:
        delivery.time_till_next_node -= self.time_step
        if not delivery.in_transition and ((delivery.is_primary) or (delivery.sharing_with is not None and delivery.current_node != delivery.sharing_with.current_node)):
          vehicle_debt[delivery.current_node] -= 1
        delivery.in_transition = True

        if delivery.time_till_next_node <= 0:
          delivery.in_transition = False
          if delivery.is_primary or (delivery.sharing_with is not None and delivery.current_node != delivery.sharing_with.current_node):
            self.vehicles[delivery.next_node] += 1
          delivery.current_node = delivery.next_node
          self.update_step(delivery)  # Update step to make sure it is correct

          if delivery.current_node == delivery.end_node:
            if delivery.sharing_with is not None:
              delivery.sharing_with.is_primary = True
              delivery.sharing_with.sharing_with = None
              delivery.sharing_with = None
            delivery.end_time = self.clock + delivery.time_till_next_node
            delivery.completed = True
            to_remove.append(delivery)

          delivery.time_till_next_node = 0


      # check for deliveries that have reached a hotspot
      if delivery.in_transition == False and delivery.completed == False:
        self.hotspot_deliveries.append(delivery)

    for delivery in to_remove:
      # if delivery.did_share:
      # print("/\\"*20, "\ndelivery arrived!") ###DEBUG
      # if delivery.did_share:  # Remove when done
      #   ###DEBUG
      #   print(f"""Delivery {delivery.id}
      #   Start Hotspot: \t {index_to_tract[delivery.start_node][-4:]}
      #   End Hotspot: \t {index_to_tract[delivery.end_node][-4:]}
      #   Same Hotspot: \t {strings_in_same_sublist(index_to_tract[delivery.start_node], index_to_tract[delivery.end_node])}
      #   Path: \t \t {[index_to_tract[n][-4:] for n in delivery.path]}
      #   Did share: \t {delivery.did_share}
      #   Shared with: \t {delivery.shared_with_final.id}
      #   Started sharing: \t {index_to_tract[delivery.shared_with_when]}"""
      #   )
      #   graph_path_delivery(delivery=delivery, show_geoids=False)  ###DEBUG

      self.active_deliveries.remove(delivery)

    if self.is_baseline:
      for i in range(self.num_hotspots):
        minimum = self.baseline_data[index_to_tract[i]]['min_time']
        maximum = self.baseline_data[index_to_tract[i]]['max_time']
        prob = [random.randint(minimum, maximum) for x in range(self.vehicles[i])]

        count = 0
        limit = 120 if len(self.deliveries)<1200 else 150
        for x in prob:
          if x<=limit: count+=1

        if (abs(vehicle_debt[i]) <= count):
          self.vehicles[i] += vehicle_debt[i]
        else:
          self.vehicles[i] = 0
          self.total_vehicles += (abs(vehicle_debt[i])-count)

    else:
      for i in range(self.num_hotspots):
        self.vehicles[i] += vehicle_debt[i]
        if self.vehicles[i] < 0:
          self.total_vehicles += -(self.vehicles[i])
          self.vehicles[i] = 0

  # adds new deliveries according to start time of deliveries : FIFO order
  def add_deliveries(self):
    for delivery in reversed(self.dormant_deliveries):
      if delivery.start_time <= self.clock:
        self.active_deliveries.append(delivery)
        self.hotspot_deliveries.append(delivery)
        self.dormant_deliveries.remove(delivery)

  # sets the next nodes of deliveries
  def set_next_nodes(self, sharing):

    if sharing:

      for delivery in self.hotspot_deliveries:

        # set for sharing
        if delivery.sharing_with is not None:

          # sharing has started
          if delivery.current_node == delivery.sharing_with.current_node:
            common_node = self.common_next(delivery, delivery.sharing_with)[0]

            # further sharing possible
            if common_node is not None:
              delivery.next_node = common_node
              delivery.path.append(common_node)
              self.update_order_rate(common_node, delivery)
              if delivery.is_primary:
                delivery.distance_traveled += self.distance_matrix[delivery.current_node][delivery.next_node]
                # self.edge_count[delivery.current_node][delivery.next_node] += 1
              delivery.time_till_next_node = self.time_matrix[delivery.current_node][delivery.next_node]
              delivery.sharing_with.time_till_next_node = delivery.time_till_next_node

            # no further sharing possible
            else:
              delivery.sharing_with.is_primary = True
              delivery.is_primary = True
              delivery.sharing_with.sharing_with = None
              delivery.sharing_with = None


          # sharing has not started and arriving at non-shared node
          elif delivery.current_node != delivery.sharing_node:
            delivery.next_node = delivery.sharing_node
            delivery.path.append(delivery.next_node)
            self.update_order_rate(delivery.next_node, delivery)
            # Add distance and time traveled to delivery total
            delivery.distance_traveled += self.distance_matrix[delivery.current_node][delivery.next_node]
            delivery.time_till_next_node = self.time_matrix[delivery.current_node][delivery.next_node]


      share_deliveries = list()

      for delivery in self.hotspot_deliveries:

        # if not sharing then add sharing pairs
        if delivery.sharing_with is None:
          share_deliveries += self.share_with(delivery)

      share_deliveries.sort()

      ## REQUEST HANDLER [start]

      for pair in share_deliveries:
        del1 = pair[1]
        del2 = pair[2]
        common_node = pair[3]

        n1 = del1.next_node if del1.in_transition else del1.current_node
        n2 = del2.next_node if del2.in_transition else del2.current_node

        clause1 = del1.sharing_with is None and del2.sharing_with is None
        clause2 = del1.end_node != n2 and del2.end_node != n1

        if clause1 and clause2:
          del1.sharing_with = del2
          del2.sharing_with = del1

          del1.num_car_changes += 1
          del2.num_car_changes += 1
          self.shares += 1


          # For debugging with deliveries
          del1.did_share = True
          del2.did_share = True

          del1.shared_with_final = del2
          del2.shared_with_final = del1

          del1.shared_with_when = n1
          del2.shared_with_when = n2
          ##############################

          del1.sharing_node = common_node
          del2.sharing_node = common_node

          del1.is_primary = True
          del2.is_primary = False

          if (not del1.in_transition):
            del1.next_node = common_node
            # self.edge_count[del1.current_node][del1.next_node] += 1
            if(del1.is_primary):
              del1.distance_traveled += self.distance_matrix[del1.current_node][del1.next_node]
            del1.path.append(common_node)
            self.update_order_rate(common_node, del1)
            del1.time_till_next_node = self.time_matrix[del1.current_node][del1.next_node]

          if (not del2.in_transition):
            del2.next_node = common_node
            # self.edge_count[del2.current_node][del2.next_node] += 1
            if(del2.is_primary):
              del2.distance_traveled += self.distance_matrix[del2.current_node][del2.next_node]
            del2.path.append(common_node)
            self.update_order_rate(common_node, del2)
            del2.time_till_next_node = self.time_matrix[del2.current_node][del2.next_node]

      ## REQUEST HANDLER [end]

    # set next nodes for those deliveries that cannot share
    for delivery in self.hotspot_deliveries:
      if delivery.sharing_with is None:
        delivery.next_node = self.next_node(delivery)
        delivery.is_primary = True
        # self.edge_count[delivery.current_node][delivery.next_node] += 1
        delivery.distance_traveled += self.distance_matrix[delivery.current_node][delivery.next_node]
        delivery.path.append(delivery.next_node)
        self.update_order_rate(delivery.next_node, delivery)
        delivery.time_till_next_node = self.time_matrix[delivery.current_node][delivery.next_node]


  # return list of deliveries to share path with as `[(Q_VALUE, DELIVERY, DELIVERY, COMMON NODE)]`
  def update_order_rate(self, node, delivery):
    self.order_rate[node] += 1
    if (delivery.start_node != node and delivery.start_node != node):
      self.non_local_order_rate[node] += 1

  def share_with(self, del1):

    pairs = list()

    # May need to change super-spot group selection to include nearby super-spots
    del_group = [ del1.current_node ] if del1.step == 2 else tract_group_lists[tract_super_dict[del1.current_node]]  # Change [ del1.current_node ] to tract_super_dict[del1.current_node] if you want to include super-spot network
    # del_group = [del1.current_node]



    for hotspot in del_group:
      for del2 in self.delivery_hotspots[hotspot]:
        if (del1 != del2 and del2.sharing_with is None and del2.next_node != del2.end_node):
          common = self.common_next(del1, del2)
          if common[0] is not None:
            pairs.append((common[1], del1, del2, common[0]))

    return pairs


  # return best next node of single delivery as `node`
  def next_node(self, delivery):
    if delivery.step == 1:  # Get to super-spot of current group
      local_Q = self.Q[tract_super_dict[delivery.current_node]]  # Q table of local group
      dest_node = tract_super_dict[delivery.current_node]  # Sets destination node to super-spot of local group

      # List if actions in local Q table given curr location and destination
      action_list = local_Q[index_to_Q[ dest_node ]][index_to_Q[ delivery.current_node ]]
      return Q_to_index[tract_super_dict[delivery.current_node]][np.argmin(action_list)]  # Get minimum Q value, then convert to global

    elif delivery.step == 2:  # In super-spot layer, get to next super-spot
      local_Q = self.Q[-1]  # Super-spot Q table
      dest_node = tract_super_dict[delivery.end_node]  # Super-spot of destination node as temp dest

      # List if actions in super-spot table given curr location and final destination
      action_list = local_Q[index_to_Q_super[ dest_node ]][index_to_Q_super[ delivery.current_node ]]
      return Q_super_to_index[np.argmin(action_list)]


    elif delivery.step == 3:  # In final group, get pathing to final destination
      local_Q = self.Q[tract_super_dict[delivery.end_node]]
      dest_node = delivery.end_node

      action_list = local_Q[index_to_Q[ dest_node ]][index_to_Q[ delivery.current_node ]]
      return Q_to_index[tract_super_dict[delivery.current_node]][np.argmin(action_list)]

    elif delivery.step == 4:  # Hotspots bordering but in different groups, travel directly to destination
      return delivery.end_node

  # Updates what step the delivery is at
  def update_step(self, delivery):
    if delivery.step == 4:
      delivery.step = 4

    elif tract_super_dict[delivery.current_node] != tract_super_dict[delivery.end_node] and delivery.current_node not in list(index_to_Q_super.keys()):  # current node super-table != dest node super-table and not at a super-spot
      delivery.step = 1

    # tract_super_dict[delivery.current_node] == delivery.current_node is TRUE
    elif tract_super_dict[delivery.current_node] != tract_super_dict[delivery.end_node] and delivery.current_node in list(index_to_Q_super.keys()):  # current node super-table != dest node super-table and is at a super-spot
      delivery.step = 2

    elif tract_super_dict[delivery.current_node] == tract_super_dict[delivery.end_node]:  # current node super-table == dest node super-table
      delivery.step = 3


  def common_next(self, del1, del2):

    node1 = del1.next_node if del1.in_transition else del1.current_node
    node2 = del2.next_node if del2.in_transition else del2.current_node

    # Make sure steps are equal
    if del1.step != del2.step:
      return (None, None)


    ###we could check here for if the share could be bad

    common_step = del1.step
    if common_step == 1:
      local_Q = self.Q[tract_super_dict[del1.current_node]]
      super_spot = tract_super_dict[node1]
      Q_value = local_Q[index_to_Q[super_spot]][index_to_Q[del1.current_node]][index_to_Q[super_spot]] + local_Q[index_to_Q[super_spot]][index_to_Q[del2.current_node]][index_to_Q[super_spot]]

      return (super_spot, Q_value)

    elif common_step == 2:
      local_Q = self.Q[-1]

      actions_del1 = local_Q[index_to_Q_super[tract_super_dict[del1.end_node]]][index_to_Q_super[node1]]
      actions_del2 = local_Q[index_to_Q_super[tract_super_dict[del2.end_node]]][index_to_Q_super[node2]]


      actions = actions_del1 + actions_del2
      actions_indexes = list(range(len(actions)))

      # Remove all actions where sharing does not benefit both deliveries
      for action in range(len(actions)):
        if self.distance_matrix[node1, tract_super_dict[del1.end_node]] <= self.distance_matrix[Q_super_to_index[action], tract_super_dict[del1.end_node]] or self.distance_matrix[node2, tract_super_dict[del2.end_node]] <= self.distance_matrix[Q_super_to_index[action], tract_super_dict[del2.end_node]]:
          actions_indexes.remove(action)

      min_Q = 10000000
      best_action = None
      for idx, Q_value in enumerate(actions):
        if idx in actions_indexes:
          if Q_value < min_Q:
            min_Q = Q_value
            best_action = idx

      return (Q_super_to_index[best_action], min_Q) if best_action is not None else (None, None)

    elif common_step == 3:
      local_Q = self.Q[tract_super_dict[del1.end_node]]

      actions_del1 = local_Q[index_to_Q[del1.end_node]][index_to_Q[node1]]
      actions_del2 = local_Q[index_to_Q[del2.end_node]][index_to_Q[node2]]
      actions = actions_del1 + actions_del2
      actions_indexes = list(range(len(actions)))  # List of possible actions

      # get a list of actions that have already been taken so that they can be removed from consideration
      for action in tract_group_lists[tract_super_dict[del1.end_node]]:
        # If actions has already been traversed OR action is not beneficial to both deliveries...
        if action in del1.path or (self.distance_matrix[node1, del1.end_node] <= self.distance_matrix[action, del1.end_node] or self.distance_matrix[node2, del2.end_node] <= self.distance_matrix[action, del2.end_node]):
          actions_indexes.remove(index_to_Q[action])

      # If no actions return none
      if len(actions_indexes) == 0:
        return (None, None)

      min_Q = 10000000
      best_action = None
      for idx, Q_value in enumerate(actions):
        if idx in actions_indexes:
          if Q_value < min_Q:
            min_Q = Q_value
            best_action = idx

      return (Q_to_index[tract_super_dict[del1.end_node]][best_action], min_Q) if best_action is not None else (None, None)

    else:  # This will prevent step 4 from sharing.
      return (None, None)


  def reset(self):
    self.deliverylist.reset_deliveries()
    self.__init__(deliverylist=self.deliverylist, num_hotspots=self.num_hotspots, distance_matrix=self.distance_matrix, time_matrix=self.time_matrix, Q_table=self.Q)




  def calculate_vehicle_change_times(self, fixed=True, factorMin=15, factorMax=30): #fixed means it will alwyas use the minimum factor while if it is not fixed it will randomly choose an integer value between factorMin and factorMax inclusive
    vehicle_change_times = dict()

    for delivery in self.deliveries:
      if(fixed):
        vehicle_change_times[delivery] = delivery.num_car_changes * factorMin
      else:
        tot = 0
        for i in range(delivery.num_car_changes):
          tot += random.randint(factorMin, factorMax)

        vehicle_change_times[delivery] = tot

    return vehicle_change_times


  def check_delivery_timelimit_success(self, delivery, time_val):
    # IMPORTANT: I am defining these manually assuming they will not change
    #  and that hours = 1.
    total_seconds = 3600*self.deliverylist.hours
    sd = self.deliverylist.sigma/60.0  # convert standard deviation in minutes to hours

    # This determines which peak the delivery is closest to. Then it gets the time difference between that
    #  peak and the delivery starting time.
    peak_diff = min(abs(total_seconds * peak - delivery.start_time) for peak in self.deliverylist.peaks)

    # Now we determine how many std. deviations away from that peak it is closer to.
    num_sd_away = peak_diff / (total_seconds*sd)

    # The closer to the peak, the more time we want to give a delivery.
    # In our formulation for time limit L, we will use the equation L = T * max{1, (2 - S)}, where T is the original delivery
    #   time limit and S how many standard deviations away the delivery start time is from its closest peak time. So when a
    #   delivery is the max possible standard deviations away from a peak, L = T. Otherwise, it will be boosted as it approaches
    #   peak hours.

    time_limit = delivery.time_limit * max(1, (2 - num_sd_away))

    return time_val < time_limit

  def results(self):
    with open(data_dir + "/avg_hotspot_data.json") as f:
      hotspot = json.load(f)

    distances = list()
    distances_tagged = list()
    times = list()
    times_tagged = list()
    wait = 0
    hops = 0
    vehicle_change_times = self.calculate_vehicle_change_times(self)

    for delivery in self.deliveries:
      t = delivery.end_time - delivery.start_time + hotspot[index_to_tract[delivery.end_node]]['time'] + hotspot[index_to_tract[delivery.start_node]]['time'] + vehicle_change_times[delivery]
      if self.check_delivery_timelimit_success(delivery, t):
        delivery.successful = True
        self.destination_success[delivery.start_node] += 1
        self.source_success[delivery.end_node] += 1

    for delivery in self.deliveries:
      t = delivery.end_time - delivery.start_time + hotspot[index_to_tract[delivery.end_node]]['time'] + hotspot[index_to_tract[delivery.start_node]]['time'] + vehicle_change_times[delivery]
      d = delivery.distance_traveled + hotspot[index_to_tract[delivery.start_node]]['distance'] + hotspot[index_to_tract[delivery.end_node]]['distance']

      distances.append( d )
      distances_tagged.append( (d, delivery.did_share, delivery.successful) )
      times.append( t )
      times_tagged.append( (t, delivery.did_share, delivery.successful) )

      if delivery.successful:

        hops += len(delivery.path) - 1

        path_time = 0
        for i in range(len(delivery.path)-1):
          path_time += self.time_matrix[delivery.path[i]][delivery.path[i+1]]

        wait += (delivery.end_time - delivery.start_time) + vehicle_change_times[delivery] - path_time

    successful_deliveries = [delivery for delivery in self.deliveries if delivery.successful]
    successful_delivery_count = len(successful_deliveries)
    distance = sum(distances)

    data = dict()
    data['shares'] = self.shares
    data['success_rate'] = successful_delivery_count / len(self.deliveries)
    data['success_rate_grouped'] = dict.fromkeys(INTERVALS, 0)

    for interval in INTERVALS:
      interval_delivs = [d for d in self.deliveries if d.time_limit == interval]
      interval_successfuls = [d for d in interval_delivs if d.successful]
      data['success_rate_grouped'][interval] = len(interval_successfuls) / max(1, len(interval_delivs))

    data['description'] = "results"
    data['sharing'] = True
    data['distances'] = distances
    data['distances_tagged'] = distances_tagged
    data['times'] = times
    data['times_tagged'] = times_tagged
    data['wait'] = wait/len(successful_deliveries)
    data['hops'] = (hops+2*len(successful_deliveries))/len(successful_deliveries)
    data['hops_all'] = (hops+2*len(self.deliveries)) / float(len(self.deliveries))  # I think this is correct?
    data['CDV'] = int(self.total_vehicles)
    data['PDV'] = int(self.calculate_PDV())
    data['destination_success'] = self.destination_success
    data['source_success'] = self.source_success
    data['order_rate'] = self.order_rate
    data['non_local_order_rate'] = self.non_local_order_rate

    return data

  # No pathsharing, uses hotspots, no super-spots
  def baseline1(self):
    self.reset()

    with open(data_dir + "/avg_hotspot_data.json") as f:
      hotspot = json.load(f)


    distances = list()
    distances_tagged = list()
    times = list()
    times_tagged = list()

    delivery_success_grouped = {interval: [0, 0] for interval in INTERVALS}

    # Use distance/time matrices that are accurate for whole city.
    successful_delivery_count = 0
    for delivery in self.deliveries:

      interval_greater = np.array(INTERVALS) >= delivery.time_limit
      grouped_index = INTERVALS[np.argmax(interval_greater)]

      delivery_success_grouped[grouped_index][1] += 1

      distance_val = self.distance_matrix[delivery.start_node][delivery.end_node] + hotspot[index_to_tract[delivery.start_node]]["distance"] + hotspot[index_to_tract[delivery.end_node]]["distance"]
      time_val = self.time_matrix[delivery.start_node][delivery.end_node] + hotspot[index_to_tract[delivery.start_node]]["time"] + hotspot[index_to_tract[delivery.end_node]]["time"]

      delivery_timelim_success = bool(self.check_delivery_timelimit_success(delivery, time_val))

      if delivery_timelim_success:
        successful_delivery_count += 1
        delivery_success_grouped[grouped_index][0] += 1

      distances.append(distance_val)
      times.append(time_val)

      distances_tagged.append( (distance_val, False, delivery_timelim_success) )
      times_tagged.append( (time_val, False, delivery_timelim_success) )


    self.reset()
    self.run(False)

    data = dict()
    data['shares'] = 0
    data['success_rate'] = successful_delivery_count / len(self.deliveries)
    data['success_rate_grouped'] = {k: v[0] / max(1, v[1]) for k, v in delivery_success_grouped.items()}
    data['description'] = "uses hotspots, does not use super-spots"
    data['sharing'] = False
    data['distances'] = distances
    data['distances_tagged'] = distances_tagged
    data['times'] = times
    data['times_tagged'] = times_tagged
    data['wait'] = 0
    data['hops'] = 3
    data['hops_all'] = 3
    data['CDV'] = int(self.total_vehicles)
    data['PDV'] = int(self.calculate_PDV())
    return data

  def baseline1_fast(self):
    with open(data_dir + "/avg_hotspot_data.json") as f:
      hotspot = json.load(f)


    distances = list()
    distances_all = list()
    times = list()
    times_all = list()

    # Use distance/time matrices that are accurate for whole city.
    delivery_number = 0
    successful_delivery_count = 0
    for delivery in self.deliveries:
      delivery_number += 1

      distance_val = self.distance_matrix[delivery.start_node][delivery.end_node] + hotspot[index_to_tract[delivery.start_node]]["distance"] + hotspot[index_to_tract[delivery.end_node]]["distance"]
      time_val = self.time_matrix[delivery.start_node][delivery.end_node] + hotspot[index_to_tract[delivery.start_node]]["time"] + hotspot[index_to_tract[delivery.end_node]]["time"]

      if self.check_delivery_timelimit_success(delivery, time_val):
        successful_delivery_count += 1

        distances.append(distance_val)
        times.append(time_val)

      distances_all.append(distance_val)
      times_all.append(time_val)

    data = dict()
    data['shares'] = 0
    data['success_rate'] = successful_delivery_count / delivery_number
    data['description'] = "uses hotspots, does not use super-spots"
    data['sharing'] = False
    data['distances'] = distances
    data['distances_all'] = distances_all
    data['total_distance_all'] = sum(distances_all)/1000  # Total distance for all including failed in km
    data['total_distance'] = sum(distances)/1000  # Total distance in km
    data['mean_distance'] = (sum(distances)/1000)/successful_delivery_count  # Mean distance in km
    data['median_distance'] = np.median(distances)/1000  # Median distance in km
    data['times'] = times
    data['mean_time'] = round( sum(times)/successful_delivery_count, 2)
    data['median_time'] = np.median(times)
    data['wait'] = 0
    data['hops'] = 3
    return data


  # No pathsharing, uses hotspots, uses super-spots
  def baseline2(self):
    self.reset()

    with open(data_dir + "/avg_hotspot_data.json") as f:
      hotspot = json.load(f)

    distances = list()
    distances_tagged = list()
    times = list()
    times_tagged = list()

    delivery_success_grouped = {interval: [0, 0] for interval in INTERVALS}

    total_hops = 0
    total_hops_all = 0  # Regardless of success
    successful_delivery_count = 0
    for delivery in self.deliveries:
      delivery_success_grouped[delivery.time_limit][1] += 1

      # Distances and times within census tract added
      distance_val = hotspot[index_to_tract[delivery.start_node]]["distance"] + hotspot[index_to_tract[delivery.end_node]]["distance"]  # Set to average hotspot values for start and end
      time_val = hotspot[index_to_tract[delivery.start_node]]["time"] + hotspot[index_to_tract[delivery.end_node]]["time"]  # Same here

      # If in same group, just get direct path distance/time
      if tract_super_dict[delivery.start_node] == tract_super_dict[delivery.end_node] or delivery.step == 4:
        distance_val += self.distance_matrix[delivery.start_node][delivery.end_node]
        time_val += self.time_matrix[delivery.start_node][delivery.end_node]

        hops_to_add = 3

      # Otherwise get distance/time from steps
      else:
        distance_val += self.distance_matrix[delivery.start_node][tract_super_dict[delivery.start_node]]
        time_val += self.time_matrix[delivery.start_node][tract_super_dict[delivery.start_node]]

        distance_val += self.distance_matrix[tract_super_dict[delivery.start_node]][tract_super_dict[delivery.end_node]]
        time_val += self.time_matrix[tract_super_dict[delivery.start_node]][tract_super_dict[delivery.end_node]]

        distance_val += self.distance_matrix[tract_super_dict[delivery.end_node]][delivery.end_node]
        time_val += self.time_matrix[tract_super_dict[delivery.end_node]][delivery.end_node]

        # Compute hops
        hops_to_add = 5
        if delivery.start_node == tract_super_dict[delivery.start_node]:
          hops_to_add -= 1
        if delivery.end_node == tract_super_dict[delivery.end_node]:
          hops_to_add -= 1

      delivery_timelim_success = bool(self.check_delivery_timelimit_success(delivery, time_val))
      if delivery_timelim_success:
        delivery_success_grouped[delivery.time_limit][0] += 1
        total_hops += hops_to_add
        successful_delivery_count += 1

      distances.append(distance_val)
      times.append(time_val)

      distances_tagged.append( (distance_val, False, delivery_timelim_success) )
      times_tagged.append( (time_val, False, delivery_timelim_success) )

      total_hops_all += hops_to_add



    self.reset()
    self.run(False)

    data = dict()
    data['shares'] = 0
    data['success_rate'] = successful_delivery_count / len(self.deliveries)
    data['success_rate_grouped'] = {k: v[0] / max(1, v[1]) for k, v in delivery_success_grouped.items()}
    data['description'] = "uses hotspots, uses super-spots"
    data['sharing'] = False
    data['distances'] = distances
    data['distances_tagged'] = distances_tagged
    data['times'] = times
    data['times_tagged'] = times_tagged
    data['wait'] = 0
    data['hops'] = float(total_hops) / float(successful_delivery_count)
    data['hops_all'] = float(total_hops_all) / float(len(self.deliveries))
    data['CDV'] = int(self.total_vehicles)
    data['PDV'] = int(self.calculate_PDV())
    return data


  def calculate_PDV(self):

    with open(data_dir + "/avg_hotspot_data.json") as f:
      data = json.load(f)

    PDV_count = 0
    tolerance = 2*self.time_step

    for hotspot in range(self.num_hotspots):
      events = []
      for delivery in self.deliveries:
        if (delivery.start_node == hotspot):
          events.append((delivery.start_time - data[index_to_tract[hotspot]]["time"] + tolerance, 1))
          events.append((delivery.start_time - tolerance, -1))
        elif (delivery.end_node == hotspot):
          events.append((delivery.end_time + tolerance, 1))
          events.append((delivery.end_time + data[index_to_tract[hotspot]]["time"] - tolerance, -1))

      # Sort the events list based on time, considering both start and end times
      events.sort()

      # Iterate through the events list to find the maximum number of active deliveries
      max_active_deliveries = 0
      active_deliveries = 0
      for _, delta in events:
        active_deliveries += delta
        max_active_deliveries = max(max_active_deliveries, active_deliveries)

      PDV_count += max_active_deliveries

    return PDV_count

## Simulate

First, define the parameters of the delivery set you are going to be using in this simulation.

In [ ]:
distance_based = "random"
load = 300
percent_max_dist = 100
hours = 1  # By default
peaks = [0.25, 0.75]  # By default
sigma = 10  # By default
check_OMSD = True

### Determine Optimal Minimum Sharing Distance

In [ ]:
if(check_OMSD):
  deliveries_graphing_dir =  data_dir + "/Graphing"
  try:
    os.mkdir(deliveries_graphing_dir)
  except:
    print(deliveries_graphing_dir + " Already Exists")

  percs_to_check = range(101)

In [ ]:
if(check_OMSD):
  res_dist = list()
  base1_dist = list()
  diff = list()


  ## UNCOMMENT THE BELOW IF YOU HAVEN'T GENERATED ALL THE GRAPH DELIVERY SETS YET FOR ANY RATIO IN THE CITY!
  deliv_list = DeliveryList(distance_based=distance_based, load=load, hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=percent_max_dist)

  deliv_list = DeliveryList.load_deliveries(
    DeliveryList.gen_deliveries_id(distance_based=distance_based, load=load,
      hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=percent_max_dist)
  )
  ###########################################


  for i in tqdm(percs_to_check):

    deliveries_id = DeliveryList.gen_deliveries_id(distance_based=distance_based,
        load=load, hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=i)

    if deliveries_id not in os.listdir(deliveries_graphing_dir):
      new_deliv_list = deliv_list.copy()
      new_deliv_list.reset_deliveries(i)

      DeliveryList.save_deliveries(new_deliv_list, graph=True)

    else:
      new_deliv_list = DeliveryList.load_deliveries(deliveries_id, graph=True)
      new_deliv_list.reset_deliveries()


    env = Environment(new_deliv_list, num_hotspots, full_dist_matrix, full_time_matrix, Q_tables)
    env.run()

    # Get results
    results = env.results()
    res_dist.append(sum(results['distances'])/1000)

    # Get baseline data
    if base1_dist == []:
      baseline1 = env.baseline1_fast()
      base1_dist.append(baseline1['total_distance_all'])
    else:
      base1_dist.append(base1_dist[-1])

  print()

  with open(deliveries_graphing_dir + f"/sim_results", 'wb') as fp:
    pickle.dump(res_dist, fp)
  with open(deliveries_graphing_dir + f"/baseline_results", 'wb') as fp:
    pickle.dump(base1_dist, fp)

  # Get smallest percentage
  min_perc = percs_to_check[np.argmin(res_dist)]

  # Load appropriate delivery set
  percent_max_dist = min_perc
  deliveries_temp = DeliveryList.load_deliveries(
    DeliveryList.gen_deliveries_id(distance_based=distance_based, load=load,
      hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=min_perc), graph=True
  )

  # Save in main folder
  deliveries = deliveries_temp.copy()
  DeliveryList.save_deliveries(deliveries, graph=False)


  for i in range(len(percs_to_check)):
    print(i, end="\t")
  print()

  for i in range(len(percs_to_check)):
    if (res_dist[i] < base1_dist[i]):
      print("🟩", end="\t")
    elif (res_dist[i] == base1_dist[i]):
      print("🟨", end="\t")
    else:
      print("🟥", end="\t")
  print()

Generating ID...
done.

Generating distribution...
done.

Generating deliveries...


NameError: name 'num_hotspots' is not defined

In [ ]:
if(check_OMSD):
  with open (deliveries_graphing_dir + f"/sim_results", 'rb') as fp:
    res_dist = pickle.load(fp)

  with open (deliveries_graphing_dir + f"/baseline_results", 'rb') as fp:
    base1_dist = pickle.load(fp)

  # Plot for each time step
  time = np.linspace(0, 100, len(percs_to_check))

  plt.figure(figsize=(10,6))
  plt.plot(percs_to_check, res_dist, label='res_dist', color='b')
  plt.plot(percs_to_check, base1_dist, label='base1_dist', color='r')

  min_perc = percs_to_check[np.argmin(res_dist)]
  plt.vlines(x = min_perc, ymin = min(res_dist)-100, ymax = max(res_dist)+100, colors = 'purple', linestyle = ':')
  plt.text(x=min_perc+2, y=max(res_dist)-50, s=f'{min_perc}%', fontsize = 14, color='darkred')


  # Adding labels and title
  #plt.ylim(bottom=0)
  plt.xlabel('Minimum Percentage of Max Distance to Allow Order Sharing (%)')
  plt.ylabel('Total Distance (kilometers)')
  plt.title('Minimum Percentage vs Total Distance for Baseline 1 and Simulation')
  plt.legend()
  plt.grid(True)

  try:
    os.mkdir(data_dir + "/Images")
  except:
    print(data_dir + "/Images" + " Already Exists")

  try:
    os.mkdir(f'{data_dir}/Images/Data-{min_children}-{superspot_hotspot_ratio}')
  except:
    print(f'{data_dir}/Images/Data-{min_children}-{superspot_hotspot_ratio}', "Already Exists")

  plt.savefig(f'{data_dir}/Images/Data-{min_children}-{superspot_hotspot_ratio}/min_perc_viability_graph.png')

  plt.show()

  print("Min:", min_perc)

### Final Simulation

In [ ]:
# If you ran the above code section 'Determine Optimal Minimum...' then this should be the correct value already, do not reinitialize it.
#   Otherwise check the spreadsheet and set the value here accordingly.
if(not check_OMSD):
  percent_max_dist = 29

# deliv_list = DeliveryList(distance_based=distance_based, load=load, hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=percent_max_dist)

deliveries = DeliveryList.load_deliveries(
  DeliveryList.gen_deliveries_id(distance_based=distance_based, load=load,
    hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=percent_max_dist), graph=False
)
# deliveries.reset_deliveries()

Loading from ./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Deliveries/0-0300-001-010-[0.25, 0.75]-029


FileNotFoundError: [Errno 2] No such file or directory: './gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Deliveries/0-0300-001-010-[0.25, 0.75]-029'

In [ ]:
env = Environment(deliverylist=deliveries, num_hotspots=num_hotspots, distance_matrix=full_dist_matrix, time_matrix=full_time_matrix, Q_table=Q_tables)
env.run()

In [ ]:
# @title Graph Path Delivery Function | graph_path_delivery(delivery, geoid_trunc=4, scale=1, show_hotspots=True, show_path=True, show_geoids=True, leg_pos='upper left')
def graph_path_delivery(delivery, geoid_trunc=4, scale=1, show_hotspots=True, show_path=True, show_geoids=True, leg_pos='upper left'):
  # Make graph df with all necessary info
  graph_df = census_df.merge(gh_df, on='GEOID', how='left')

  # Plot grpah base, grey
  fig, ax = plt.subplots(figsize=(int(14*scale), int(16*scale)))
  graph_df.plot(ax=ax, color='#e6e6e6', edgecolor='black')

  # Function to merge children geometries for super tracts
  def merge_children_geometries(row, gdf):
    children_ids = ast.literal_eval(row['children']) if 'children' in row else []
    children_geometries = gdf[(gdf['GEOID'].isin(children_ids)) | (gdf['GEOID'] == row['GEOID'])]
    if not children_geometries.empty:
      merged_geometry = children_geometries.unary_union
      return merged_geometry
    return None


  super_df = graph_df[graph_df['is_super']]


  # Plot all individual census tracts with light borders on top
  graph_df.plot(ax=ax, color='#e6e6e6', edgecolor='none')
  graph_df.plot(ax=ax, color='none', edgecolor='#aaaaaa', linewidth=0.5)

  # Plot superspots with darker fill and white borders
  for _, row in super_df.iterrows():
    gpd.GeoSeries([row['geometry']]).plot(ax=ax, color='#999999', edgecolor="#f5f5f5", linewidth=1.5)

  # Plot merged group geometries with thick black borders (no fill)
  for _, row in super_df.iterrows():
    merged_geom = merge_children_geometries(row, graph_df)
    if merged_geom:
      gpd.GeoSeries([merged_geom]).plot(ax=ax, color='none', edgecolor='black', linewidth=2)


  # Plot hotspots
  if show_hotspots:
    ax.scatter(graph_df['x'], graph_df['y'], color='black', s=8*scale*scale)


  ax.set_facecolor('#abe6ff')


  paths = [delivery.path, []]
  shared_paths = []
  if delivery.did_share:
    paths[1] = delivery.shared_with_final.path

    pairs_list1 = [(paths[0][i], paths[0][i+1]) for i in range(len(paths[0])-1)]
    pairs_list2 = [(paths[1][i], paths[1][i+1]) for i in range(len(paths[1])-1)]

    shared_paths = [pair for pair in pairs_list1 if pair in pairs_list2]


  # Plot path of delivery
  if show_path:
    for shared, path in enumerate(paths):
      if path:
        for i in range(len(path) - 1):  # Iterate through path
          start_point = graph_df[graph_df['GEOID'] == index_to_tract[path[i]]][['x', 'y']].iloc[0]  # Get start point of path
          end_point = graph_df[graph_df['GEOID'] == index_to_tract[path[i+1]]][['x', 'y']].iloc[0]  # Get end point of path

          if (path[i], path[i+1]) in shared_paths:
            arrow_color = 'green' # Green (shared path)
          elif shared == 1:
            arrow_color = '#ffb700' # Orange (other delivery)
          else:
            arrow_color = 'blue' # Blue (main delivery)

          # Draw line
          ax.annotate('', xy=(end_point.x, end_point.y), xytext=(start_point.x, start_point.y), arrowprops=dict(arrowstyle="->", color=arrow_color, linewidth=4))

  start_marker = mlines.Line2D([], [], color='#ffb3ff', marker='o', linestyle='None', markersize=10, markeredgecolor='black', label='Start Hotspot')
  start_sharing_marker = mlines.Line2D([], [], color='#9900cc', marker='o', linestyle='None', markersize=10, markeredgecolor='black', label='Start Hotspot [Sharing]')
  end_sharing_marker = mlines.Line2D([], [], color='#008000', marker='o', linestyle='None', markersize=10, markeredgecolor='black', label='End Hotspot [Sharing]')
  end_marker = mlines.Line2D([], [], color='lightgreen', marker='o', linestyle='None', markersize=10, markeredgecolor='black', label='End Hotspot')

  handles_list = [start_marker, end_marker]
  if delivery.did_share:
    if paths[1][0] != paths[0][0]:
      ax.scatter(graph_df[graph_df['GEOID'] == index_to_tract[paths[1][0]]]['x'], graph_df[graph_df['GEOID'] == index_to_tract[paths[1][0]]]['y'], color='#9900cc', edgecolor='black', s=40*scale*scale)  # Plot intermediate start hotspot
      handles_list.append(start_sharing_marker)

    if paths[1][-1] != paths[0][-1]:
      ax.scatter(graph_df[graph_df['GEOID'] == index_to_tract[paths[1][-1]]]['x'], graph_df[graph_df['GEOID'] == index_to_tract[paths[1][-1]]]['y'], color='#008000', edgecolor='black', s=40*scale*scale)  # Plot intermediate end hotspot
      handles_list.append(end_sharing_marker)


  ax.legend(handles=handles_list, fontsize=20, loc=leg_pos, frameon=True)

  ax.scatter(graph_df[graph_df['GEOID'] == index_to_tract[paths[0][0]]]['x'], graph_df[graph_df['GEOID'] == index_to_tract[paths[0][0]]]['y'], color='#ffb3ff', edgecolor='black', s=40*scale*scale)  # Plot start hotspot
  ax.scatter(graph_df[graph_df['GEOID'] == index_to_tract[paths[0][-1]]]['x'], graph_df[graph_df['GEOID'] == index_to_tract[paths[0][-1]]]['y'], color='lightgreen', edgecolor='black', s=40*scale*scale)  # Plot end hotspot

  if show_geoids:
    for idx, row in graph_df.iterrows():
      ax.annotate(row['GEOID'][-1*geoid_trunc:], (row['x'], row['y']), textcoords="offset points", xytext=(5,5), ha='center', fontsize=8*scale, color='black')

  title_str = f"{city_name} - Path Taken by Delivery {delivery.id}"
  if delivery.did_share:
    title_str += f", sharing with Delivery {delivery.shared_with_final.id}"
  plt.title(title_str, size=28)
  plt.xlabel('Longitude', size=20)
  plt.ylabel('Latitude', size=20)
  plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

the_Ds = [d for d in deliveries.deliveries if d.did_share]

graph_path_delivery(the_Ds[0], scale=1.4, show_hotspots=False, show_path=False, show_geoids=False, leg_pos='upper left')

In [ ]:
results = env.results()
baseline1 = env.baseline1()
baseline2 = env.baseline2()

In [ ]:
def print_results(results, name:str="Results", show_mean=False, show_median=True, show_full=False, show_req_clock=False, show_all=False):
  successful_distances = [d[0] for d in results['distances_tagged'] if d[2]]
  successful_times = [t[0] for t in results['times_tagged'] if t[2]]

  print(name)
  print("Shares: ", results['shares'])
  print("success rate: ", results['success_rate'], '%')
  print("total distance: ", sum(successful_distances)/1000, 'km')
  if show_full or show_all:
    print("total distance (regardless fail): ", sum(results['distances'])/1000, 'km')
    print("total time (regardless fail): ", sum(results['times']), 's')
  if show_mean or show_all:
    mean = np.mean(successful_distances)/1000

    print("mean distance: ", mean, 'km')
  if show_median or show_all:
    median = np.median(successful_distances)/1000

    print("median distance: ", median, 'km')
  if show_mean or show_all:
    mean = np.mean(successful_times)

    print("mean time: ", mean, 's')
  if show_median or show_all:
    median = np.median(successful_times)

    print("median time: ", median, 's')
  print("Vehicles ", results['CDV'])
  print("Hops ", results['hops'])
  if show_full or show_all:
    print("Hops (regardless fail): ", results['hops_all'])
  print()

In [ ]:
print_results(results, show_all=True)

print_results(baseline1, name="Baseline 1", show_all=True)

print_results(baseline2, name="Baseline 2", show_all=True)

In [ ]:
def print_dict(d):
  for k, v in d.items():
    print(f"{k}: {v*100}%")
  print()

print_dict(results['success_rate_grouped'])
print_dict(baseline1['success_rate_grouped'])
print_dict(baseline2['success_rate_grouped'])

In [ ]:
# Create Directories
try:
  os.mkdir(data_dir + f"/Results/")
except:
  print(data_dir + f"/Results/" + " Already Exists")

try:
  os.mkdir(data_dir + f"/Results/Data-{min_children}-{superspot_hotspot_ratio}/")
except:
  print(data_dir + f"/Results/Data-{min_children}-{superspot_hotspot_ratio}/" + " Already Exists")


class NumpyArrayEncoder(json.JSONEncoder):
  def default(self, obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    # Check if obj is a NumPy integer and convert to Python int
    if isinstance(obj, (np.int64, np.int32, np.int16, np.int8)):
        return int(obj)
    return json.JSONEncoder.default(self, obj)

In [ ]:
with open(f'{data_dir}/Results/Data-{min_children}-{superspot_hotspot_ratio}/results_{deliveries.id}_{min_children}_{superspot_hotspot_ratio}.json', 'w') as f:
  json.dump(results, f, cls=NumpyArrayEncoder)

with open(f'{data_dir}/Results/Data-{min_children}-{superspot_hotspot_ratio}/baseline1_{deliveries.id}_{min_children}_{superspot_hotspot_ratio}.json', 'w') as f:
  json.dump(baseline1, f, cls=NumpyArrayEncoder)

with open(f'{data_dir}/Results/Data-{min_children}-{superspot_hotspot_ratio}/baseline2_{deliveries.id}_{min_children}_{superspot_hotspot_ratio}.json', 'w') as f:
  json.dump(baseline2, f, cls=NumpyArrayEncoder)

#### Repeating Simulation

In [ ]:
try:
  os.mkdir(data_dir + f"/Results/Data-{min_children}-{superspot_hotspot_ratio}/Repeating")
except:
  print(data_dir + f"/Results/Data-{min_children}-{superspot_hotspot_ratio}/Repeating" + " Already Exists")

In [ ]:
iter = 10  # How many iterations of unique delivery sets and results to generate
already_existing = 0  # Specify how many DELIVERY SETS already exist so you don't have to regenerate them (set to 0 if none generated yet)


# Loop over iter...
for i in range(iter):
  if i < already_existing:  # If delivery set already existing:
    # Get the ID
    multi_id = DeliveryList.gen_deliveries_id(distance_based=distance_based, load=load,
      hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=percent_max_dist, gen_multi=True, multi_id=i+1)
    print(multi_id, "already exists.")

    # Load the deliveries based on ID
    multi_delivs = DeliveryList.load_deliveries(multi_id)
  else:
    # Otherwise create the new set of deliveries
    multi_delivs = DeliveryList(distance_based=distance_based, load=load, hours=hours, peaks=peaks, sigma=sigma, percent_max_dist=percent_max_dist, gen_multi=True)

  # Run the environment on the delivery set
  multi_env = Environment(deliverylist=multi_delivs, num_hotspots=num_hotspots, distance_matrix=full_dist_matrix, time_matrix=full_time_matrix, Q_table=Q_tables)
  multi_env.run()

  # Get all relevant results and baselines
  multi_results = multi_env.results()
  multi_baseline1 = multi_env.baseline1()
  multi_baseline2 = multi_env.baseline2()

  # Save results in /Repeating directory
  with open(f'{data_dir}/Results/Data-{min_children}-{superspot_hotspot_ratio}/Repeating/results_{multi_delivs.id}_{min_children}_{superspot_hotspot_ratio}_{i}.json', 'w') as f:
    json.dump(multi_results, f, cls=NumpyArrayEncoder)

  with open(f'{data_dir}/Results/Data-{min_children}-{superspot_hotspot_ratio}/Repeating/baseline1_{multi_delivs.id}_{min_children}_{superspot_hotspot_ratio}_{i}.json', 'w') as f:
    json.dump(multi_baseline1, f, cls=NumpyArrayEncoder)

  with open(f'{data_dir}/Results/Data-{min_children}-{superspot_hotspot_ratio}/Repeating/baseline2_{multi_delivs.id}_{min_children}_{superspot_hotspot_ratio}_{i}.json', 'w') as f:
      json.dump(multi_baseline2, f, cls=NumpyArrayEncoder)


# Legacy Results

In [ ]:
#use generated deliveries, will be in a different format